# ΔPPL real do STRATA numa T4

Antes de rodar: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Depois *Executar tudo*.

Roda `tests/ppl/strata_ppl.py` no wikitext-2 inteiro (~140 janelas de 2048) com `unsloth/Llama-3.2-1B`, que é uma cópia sem licença restrita do `meta-llama/Llama-3.2-1B`, então não precisa de login no Hugging Face.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "claude/project-thread-2p2spc"
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
%cd strata
!git log --oneline -1

In [ ]:
!pip install -q -U transformers datasets safetensors
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs)
import torch, transformers, datasets
print(torch.__version__, transformers.__version__, datasets.__version__, torch.cuda.is_available())

Rodada curta para conferir que tudo funciona (menos de 2 minutos).

In [ ]:
!python tests/ppl/strata_ppl.py --model unsloth/Llama-3.2-1B --max-chunks 4 --depths 2,7 --int-bits 4 --out /content/smoke

Tabela completa: estratos 0 a 7 e INT4/5/6/8, wikitext-2 inteiro.

In [ ]:
!python tests/ppl/strata_ppl.py --model unsloth/Llama-3.2-1B

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('tests/ppl/resultados/unsloth__Llama-3.2-1B.md').read()))

In [ ]:
from google.colab import files
files.download('tests/ppl/resultados/unsloth__Llama-3.2-1B.md')
files.download('tests/ppl/resultados/unsloth__Llama-3.2-1B.json')

## INT8 progressivo

Um só arquivo INT8 por bloco de 32 em sinal+magnitude; cortar planos de magnitude dá 4 a 8 bits, como os estratos do STRATA. Compare com a tabela acima (mesmos tokens, mesma referência BF16).

In [ ]:
!python tests/ppl/strata_ppl.py --model unsloth/Llama-3.2-1B --depths 7 --int-bits '' --prog-bits 4,5,6,7,8 --out /content/prog
from IPython.display import Markdown, display
display(Markdown(open('/content/prog/unsloth__Llama-3.2-1B.md').read()))
from google.colab import files
files.download('/content/prog/unsloth__Llama-3.2-1B.md'); files.download('/content/prog/unsloth__Llama-3.2-1B.json')